# Feature Engineering
## Elo Ratings & Rolling Statistics

This notebook demonstrates the feature engineering process with leakage prevention.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


## Load Feature-Engineered Data


In [ ]:
df = pd.read_parquet('../data/processed/features.parquet')
print(f"Shape: {df.shape}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"\nFeature columns:")
print([col for col in df.columns if any(x in col for x in ['Elo', '_L5', 'Diff'])])


## Elo Rating Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elo distribution
axes[0].hist([df['Home_Elo'], df['Away_Elo']], bins=30, label=['Home', 'Away'], alpha=0.7)
axes[0].set_title('Elo Rating Distribution')
axes[0].set_xlabel('Elo Rating')
axes[0].legend()

# Elo difference
axes[1].hist(df['Elo_Diff'], bins=30, alpha=0.7)
axes[1].set_title('Elo Difference (Home - Away)')
axes[1].set_xlabel('Elo Difference')
axes[1].axvline(0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"Average Elo Difference: {df['Elo_Diff'].mean():.1f}")
print(f"(Positive = Home advantage)")


## Rolling Features Analysis


In [ ]:
print("Rolling Feature Statistics:")
print(df[['Home_Goals_L5', 'Away_Goals_L5', 'Home_Form_L5', 'Away_Form_L5']].describe())

# Correlation with outcome
df['Result_Numeric'] = df['FTR'].map({'H': 1, 'D': 0, 'A': -1})
correlations = df[['Elo_Diff', 'Goals_Diff_L5', 'Form_Diff_L5', 'Result_Numeric']].corr()['Result_Numeric'].drop('Result_Numeric')
print(f"\nCorrelation with match result:")
print(correlations.sort_values(ascending=False))


## Key Insights

1. **Elo Ratings**: Range from ~1320 to ~1790 (reasonable spread)
2. **Elo Difference**: Positive on average (home advantage reflected)
3. **Rolling Features**: Capture recent team form
4. **Leakage Prevention**: All features use pre-match data only

## Next Steps (M4)
- Train baseline model (Logistic Regression)
- Train advanced model (XGBoost)
- Apply probability calibration
- Evaluate on test set
